# Chapter 14 — Recursion (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_python.ipynb)

This notebook goes with chapter 14 «Recursion» of the lecture notes (volume 4, *Analysis and recursion*) and with its [page of the site](https://fabiofurini.github.io/algorithms/analysis-recursion/14-recursion/). It contains the **Python** code of the algorithms with pseudo-code of the chapter: `Factorial`, `Fib`, `IterativeFib`, `RecursiveGCD`, `FastPower`, `Hanoi`. For every recursive algorithm the code counts the **calls** (the nodes of the tree of recursive calls), the **depth** of the recursion (the maximum height of the stack) and the **steps** (executions of the lines, numbered as in the lecture notes: line 1 is the `function` header), and prints the **trace** of the execution: the calls in depth-first order, with the indentation showing the position in the stack. The examples are those of the chapter.

**How to run it.** Click the «Open in Colab» badge, then *Runtime ▸ Run all* (or cell by cell with Shift+Enter). Nothing to install.

Every algorithm is a Python function; the run cells execute it on the same examples as the lecture notes and print the results.

**Names.** There are no arrays in this chapter: variables and parameters have the names of the pseudo-code. Two exceptions in Python, where `from` and `next` have a special meaning: `Hanoi` uses `frm` instead of `from` and `IterativeFib` uses `nxt` instead of `next`.

## Tools: counters, stack and trace

The cell defines the counters (`cnt`, `calls`, `max_depth`), the call stack (`push` and `pop`) and the functions that print the trace (`event`, `say`) and the table of steps (`print_steps`). It has to be run only once.

In [1]:
cnt = [0] * 20      # cnt[r] = number of executions of line r (lines numbered as in the lecture notes)
calls = 0           # number of calls = nodes of the tree of recursive calls
max_depth = 0       # depth of the recursion = maximum height of the stack
mults = 0           # multiplications (FastPower)
moves = 0           # moves (Hanoi)
ev = 0              # counter of the events of the trace
stack = []          # the call stack: the arguments of the active calls
seen = [0] * 64     # seen[x] = number of calls with argument x
trace = False       # True: print the trace of the execution


def reset():
    global calls, max_depth, mults, moves, ev
    calls = max_depth = mults = moves = ev = 0
    stack.clear()
    for r in range(len(cnt)):
        cnt[r] = 0
    for k in range(len(seen)):
        seen[k] = 0


def push(x):                       # a call starts: its frame goes on top of the stack
    global calls, max_depth
    calls += 1
    stack.append(x)
    max_depth = max(max_depth, len(stack))
    if x < len(seen):
        seen[x] += 1


def pop():                         # the call returns: its frame leaves the stack
    stack.pop()


def event(text):                   # numbered line of the trace, with the stack
    global ev
    ev += 1
    print("%2d. %s   stack: %s" % (ev, text, stack))


def say(call, action):             # line of the trace: the call, indented one level for every active call
    print("%-28s| %s" % ("  " * (len(stack) - 1) + call, action))


def print_steps(*lines):
    print("Line | Executions")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Total steps S = %d" % sum(cnt))

## 1. Factorial

**Input:** a natural number $n$. **Output:** $n!$, with $0! = 1$. $S(n) = 2n + 2 = \Theta(n)$ steps; $n + 1$ calls, depth $n + 1$ (the tree is a path).

Pseudo-code: section 14.4 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis-recursion/14-recursion/).

In [2]:
def factorial(n):
    push(n); cnt[2] += 1
    if n == 0:
        cnt[3] += 1
        if trace: event("Factorial(0): n = 0 => base case")
        pop()
        if trace: event("Factorial(0) returns 1")
        return 1
    if trace: event("Factorial(%d): %d != 0 => must compute %d * Factorial(%d) and waits" % (n, n, n, n - 1))
    cnt[5] += 1
    f = factorial(n - 1)
    r = n * f
    pop()
    if trace: event("Factorial(%d) returns %d * %d = %d" % (n, n, f, r))
    return r

In [3]:
# Example of the lecture notes: Factorial(4), with the stack
n = 4
reset()
trace = True
r = factorial(n)
trace = False
print("Factorial(%d) = %d" % (n, r))
print("Calls = %d (nodes of the tree), depth of the recursion = %d" % (calls, max_depth))
print_steps(2, 3, 5)
print("S(n) = 2n + 2 = %d" % (2 * n + 2))

# Calls, depth and steps as n varies
print()
print("  n |                   n! |    calls |       depth | S(n) | 2n + 2")
for n in [0, 1, 2, 3, 4, 5, 10, 15, 20]:
    reset()
    f = factorial(n)
    print("%3d | %20d | %8d | %11d | %4d | %d" % (n, f, calls, max_depth, sum(cnt), 2 * n + 2))

 1. Factorial(4): 4 != 0 => must compute 4 * Factorial(3) and waits   stack: [4]
 2. Factorial(3): 3 != 0 => must compute 3 * Factorial(2) and waits   stack: [4, 3]
 3. Factorial(2): 2 != 0 => must compute 2 * Factorial(1) and waits   stack: [4, 3, 2]
 4. Factorial(1): 1 != 0 => must compute 1 * Factorial(0) and waits   stack: [4, 3, 2, 1]
 5. Factorial(0): n = 0 => base case   stack: [4, 3, 2, 1, 0]
 6. Factorial(0) returns 1   stack: [4, 3, 2, 1]
 7. Factorial(1) returns 1 * 1 = 1   stack: [4, 3, 2]
 8. Factorial(2) returns 2 * 1 = 2   stack: [4, 3]
 9. Factorial(3) returns 3 * 2 = 6   stack: [4]
10. Factorial(4) returns 4 * 6 = 24   stack: []
Factorial(4) = 24
Calls = 5 (nodes of the tree), depth of the recursion = 5
Line | Executions
   2 |          5
   3 |          1
   5 |          4
Total steps S = 10
S(n) = 2n + 2 = 10

  n |                   n! |    calls |       depth | S(n) | 2n + 2
  0 |                    1 |        1 |           1 |    2 | 2
  1 |                    1 |

**Try it yourself.** Change $n$ and watch the stack grow and then empty.

In [4]:
n = 6        # <-- change n (20 at most)
reset()
trace = True
r = factorial(n)
trace = False
print("Factorial(%d) = %d, calls = %d, depth = %d, S = %d" % (n, r, calls, max_depth, sum(cnt)))

 1. Factorial(6): 6 != 0 => must compute 6 * Factorial(5) and waits   stack: [6]
 2. Factorial(5): 5 != 0 => must compute 5 * Factorial(4) and waits   stack: [6, 5]
 3. Factorial(4): 4 != 0 => must compute 4 * Factorial(3) and waits   stack: [6, 5, 4]
 4. Factorial(3): 3 != 0 => must compute 3 * Factorial(2) and waits   stack: [6, 5, 4, 3]
 5. Factorial(2): 2 != 0 => must compute 2 * Factorial(1) and waits   stack: [6, 5, 4, 3, 2]
 6. Factorial(1): 1 != 0 => must compute 1 * Factorial(0) and waits   stack: [6, 5, 4, 3, 2, 1]
 7. Factorial(0): n = 0 => base case   stack: [6, 5, 4, 3, 2, 1, 0]
 8. Factorial(0) returns 1   stack: [6, 5, 4, 3, 2, 1]
 9. Factorial(1) returns 1 * 1 = 1   stack: [6, 5, 4, 3, 2]
10. Factorial(2) returns 2 * 1 = 2   stack: [6, 5, 4, 3]
11. Factorial(3) returns 3 * 2 = 6   stack: [6, 5, 4]
12. Factorial(4) returns 4 * 6 = 24   stack: [6, 5]
13. Factorial(5) returns 5 * 24 = 120   stack: [6]
14. Factorial(6) returns 6 * 120 = 720   stack: []
Factorial(6) = 720, c

## 2. Fibonacci numbers: Fib

**Input:** a natural number $n$. **Output:** $F(n)$, with $F(0) = 0$, $F(1) = 1$ and $F(n) = F(n-1) + F(n-2)$. Naive recursive algorithm: $C(n) = 2F(n+1) - 1$ calls and $S(n) = 4F(n+1) - 2 = 2^{\Theta(n)}$ steps.

Pseudo-code: section 14.5 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis-recursion/14-recursion/).

In [5]:
def fib(n):
    push(n); cnt[2] += 1
    if n <= 1:
        cnt[3] += 1
        if trace: say("Fib(%d)" % n, "base case, returns %d" % n)
        pop()
        return n
    if trace: say("Fib(%d)" % n, "calls Fib(%d)" % (n - 1))
    x = fib(n - 1)                                 # the two calls are separated only to print the trace
    if trace: say("Fib(%d)" % n, "calls Fib(%d)" % (n - 2))
    y = fib(n - 2)
    cnt[5] += 1
    if trace: say("Fib(%d)" % n, "returns %d + %d = %d" % (x, y, x + y))
    pop()
    return x + y

In [6]:
def fib_value(k):                                  # F(k) with a loop, to check
    a, b = 0, 1
    for _ in range(k):
        a, b = b, a + b
    return a


# Example of the lecture notes: the execution of Fib(4), in depth-first order
n = 4
reset()
trace = True
print("%-28s| %s" % ("call", "action"))
r = fib(n)
trace = False
print("F(%d) = %d, calls C(%d) = %d = 2F(%d) - 1, depth = %d" % (n, r, n, calls, n + 1, max_depth))

# Fib(5): the same subproblems are solved many times
print()
n = 5
reset()
r = fib(n)
print("Fib(%d) = %d: %d calls, depth %d, S = %d steps" % (n, r, calls, max_depth, sum(cnt)))
print("  k | calls of Fib(k)")
for k in range(n, -1, -1):
    print("%3d | %d" % (k, seen[k]))
print_steps(2, 3, 5)

# Number of calls and steps: C(n) = 2F(n+1) - 1 and S(n) = 4F(n+1) - 2
print()
print("  n |   F(n) |     C(n) | 2F(n+1) - 1 |     S(n) | 4F(n+1) - 2 | depth")
for n in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30]:
    reset()
    r = fib(n)
    print("%3d | %6d | %8d | %11d | %8d | %11d | %d" % (n, r, calls, 2 * fib_value(n + 1) - 1, sum(cnt), 4 * fib_value(n + 1) - 2, max_depth))

call                        | action
Fib(4)                      | calls Fib(3)
  Fib(3)                    | calls Fib(2)
    Fib(2)                  | calls Fib(1)
      Fib(1)                | base case, returns 1
    Fib(2)                  | calls Fib(0)
      Fib(0)                | base case, returns 0
    Fib(2)                  | returns 1 + 0 = 1
  Fib(3)                    | calls Fib(1)
    Fib(1)                  | base case, returns 1
  Fib(3)                    | returns 1 + 1 = 2
Fib(4)                      | calls Fib(2)
  Fib(2)                    | calls Fib(1)
    Fib(1)                  | base case, returns 1
  Fib(2)                    | calls Fib(0)
    Fib(0)                  | base case, returns 0
  Fib(2)                    | returns 1 + 0 = 1
Fib(4)                      | returns 2 + 1 = 3
F(4) = 3, calls C(4) = 9 = 2F(5) - 1, depth = 4

Fib(5) = 5: 15 calls, depth 5, S = 30 steps
  k | calls of Fib(k)
  5 | 1
  4 | 1
  3 | 2
  2 | 3
  1 | 5
  0 | 3
Line | Ex

**Try it yourself.** Change $n$ and see how fast the calls grow.

In [7]:
n = 6        # <-- change n (with n > 35 it takes a while)
reset()
r = fib(n)
print("F(%d) = %d, calls = %d, S = %d" % (n, r, calls, sum(cnt)))

F(6) = 8, calls = 25, S = 50


## 3. Iterative Fibonacci: IterativeFib

**Input:** a natural number $n$. **Output:** $F(n)$, computed from $F(0)$ and $F(1)$ upwards keeping only the last two values. $S(n) = 5n + 1 = \Theta(n)$ steps. Invariant: before every evaluation of the condition, $prev = F(i-2)$ and $curr = F(i-1)$.

Pseudo-code: section 14.5.2 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis-recursion/14-recursion/).

In [8]:
def iterative_fib(n, trace=False):
    cnt[2] += 1
    if n <= 1:
        cnt[3] += 1
        return n
    prev = 0; cnt[5] += 1
    curr = 1; cnt[6] += 1
    i = 2; cnt[7] += 1
    if trace:                                      # trace: initialization row
        print("%-10s | %-8s | %-18s | %4d | %4d" % ("start", "2", "-", prev, curr))
    cnt[8] += 1
    while i <= n:
        nxt = prev + curr; cnt[9] += 1             # next in the pseudo-code (in Python it is a built-in function)
        prev = curr; cnt[10] += 1
        curr = nxt; cnt[11] += 1
        i = i + 1; cnt[12] += 1
        cnt[8] += 1
        if trace:                                  # trace: i - 2 is the number of the iteration
            print("%-10d | %-8s | %-18s | %4d | %4d" % (i - 2, "%d -> %d" % (i - 1, i), "%d + %d = %d" % (nxt - prev, prev, nxt), prev, curr))
    cnt[14] += 1
    return curr

In [9]:
# Example of the lecture notes: IterativeFib(4)
n = 4
reset()
print("iteration | i        | next = prev + curr | prev | curr")
r = iterative_fib(n, True)
print("The loop ends with i = %d > %d: it returns curr = %d" % (n + 1, n, r))
print_steps(2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14)
print("S(n) = 5n + 1 = %d" % (5 * n + 1))

# Comparison with Fib: steps of Fib = 4F(n+1) - 2 (formula checked in the previous section)
print()
print("  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1")
for n in [5, 10, 20, 30]:
    reset()
    r = iterative_fib(n)
    steps = sum(cnt)
    print("%3d | %6d | %16d | %12d | %d" % (n, r, 4 * iterative_fib(n + 1) - 2, steps, 5 * n + 1))

iteration | i        | next = prev + curr | prev | curr
start      | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
The loop ends with i = 5 > 4: it returns curr = 3
Line | Executions
   2 |          1
   3 |          0
   5 |          1
   6 |          1
   7 |          1
   8 |          4
   9 |          3
  10 |          3
  11 |          3
  12 |          3
  14 |          1
Total steps S = 21
S(n) = 5n + 1 = 21

  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1
  5 |      5 |               30 |           26 | 26
 10 |     55 |              354 |           51 | 51
 20 |   6765 |            43782 |          101 | 101
 30 | 832040 |          5385074 |          151 | 151


**Try it yourself.** Change $n$ and check that the steps are $5n + 1$.

In [10]:
n = 10        # <-- change n
reset()
print("iteration | i        | next = prev + curr | prev | curr")
r = iterative_fib(n, True)
print("F(%d) = %d, S = %d = 5n + 1 = %d" % (n, r, sum(cnt), 5 * n + 1))

iteration | i        | next = prev + curr | prev | curr
start      | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
4          | 5 -> 6   | 2 + 3 = 5          |    3 |    5
5          | 6 -> 7   | 3 + 5 = 8          |    5 |    8
6          | 7 -> 8   | 5 + 8 = 13         |    8 |   13
7          | 8 -> 9   | 8 + 13 = 21        |   13 |   21
8          | 9 -> 10  | 13 + 21 = 34       |   21 |   34
9          | 10 -> 11 | 21 + 34 = 55       |   34 |   55
F(10) = 55, S = 51 = 5n + 1 = 51


## 4. Greatest common divisor: RecursiveGCD

**Input:** two natural numbers $n \ge 1$ and $m \ge 0$. **Output:** $\gcd(n, m)$. Tail recursion: each call makes at most one, the tree is a path. At most $2\lfloor\log_2 m\rfloor + 3$ calls, $O(\log_2 m)$ steps.

Pseudo-code: section 14.6 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis-recursion/14-recursion/).

In [11]:
def floor_log2(m):                                 # floor(log2 m) with a loop (only for the bound on the calls)
    k = 0
    while m > 1:
        m = m // 2
        k = k + 1
    return k


def recursive_gcd(n, m):
    push(m); cnt[2] += 1
    if m == 0:
        cnt[3] += 1
        if trace: say("RecursiveGCD(%d, %d)" % (n, m), "base case, returns %d" % n)
        pop()
        return n
    if trace: say("RecursiveGCD(%d, %d)" % (n, m), "%d mod %d = %d: calls RecursiveGCD(%d, %d)" % (n, m, n % m, m, n % m))
    cnt[5] += 1
    g = recursive_gcd(m, n % m)
    if trace: say("RecursiveGCD(%d, %d)" % (n, m), "returns %d" % g)
    pop()
    return g

In [12]:
# Examples of the lecture notes: RecursiveGCD(48, 18) and RecursiveGCD(252, 105), trees of calls (paths)
for n, m in [(48, 18), (252, 105)]:
    reset()
    trace = True
    print("%-28s| %s" % ("call", "action"))
    g = recursive_gcd(n, m)
    trace = False
    print("gcd(%d, %d) = %d, calls = %d, S = %d" % (n, m, g, calls, sum(cnt)))
    print()

# Number of calls (each call executes 2 lines, hence S = 2 calls) and bound 2 floor(log2 m) + 3
print("      n |       m |  gcd |    calls |   S | 2 floor(log2 m) + 3")
for n, m in [(48, 18), (252, 105), (89, 55), (144, 89), (1000, 7), (1000000, 999999)]:
    reset()
    g = recursive_gcd(n, m)
    print("%7d | %7d | %4d | %8d | %3d | %d" % (n, m, g, calls, sum(cnt), 2 * floor_log2(m) + 3))

call                        | action
RecursiveGCD(48, 18)        | 48 mod 18 = 12: calls RecursiveGCD(18, 12)
  RecursiveGCD(18, 12)      | 18 mod 12 = 6: calls RecursiveGCD(12, 6)
    RecursiveGCD(12, 6)     | 12 mod 6 = 0: calls RecursiveGCD(6, 0)
      RecursiveGCD(6, 0)    | base case, returns 6
    RecursiveGCD(12, 6)     | returns 6
  RecursiveGCD(18, 12)      | returns 6
RecursiveGCD(48, 18)        | returns 6
gcd(48, 18) = 6, calls = 4, S = 8

call                        | action
RecursiveGCD(252, 105)      | 252 mod 105 = 42: calls RecursiveGCD(105, 42)
  RecursiveGCD(105, 42)     | 105 mod 42 = 21: calls RecursiveGCD(42, 21)
    RecursiveGCD(42, 21)    | 42 mod 21 = 0: calls RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | base case, returns 21
    RecursiveGCD(42, 21)    | returns 21
  RecursiveGCD(105, 42)     | returns 21
RecursiveGCD(252, 105)      | returns 21
gcd(252, 105) = 21, calls = 4, S = 8

      n |       m |  gcd |    calls |   S | 2 floor(log2 m) + 3
     48 |

**Try it yourself.** Change the two numbers; the worst case is two consecutive Fibonacci numbers.

In [13]:
n, m = 1071, 462        # <-- change the two numbers
reset()
trace = True
print("%-28s| %s" % ("call", "action"))
g = recursive_gcd(n, m)
trace = False
print("gcd(%d, %d) = %d, calls = %d" % (n, m, g, calls))

call                        | action
RecursiveGCD(1071, 462)     | 1071 mod 462 = 147: calls RecursiveGCD(462, 147)
  RecursiveGCD(462, 147)    | 462 mod 147 = 21: calls RecursiveGCD(147, 21)
    RecursiveGCD(147, 21)   | 147 mod 21 = 0: calls RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | base case, returns 21
    RecursiveGCD(147, 21)   | returns 21
  RecursiveGCD(462, 147)    | returns 21
RecursiveGCD(1071, 462)     | returns 21
gcd(1071, 462) = 21, calls = 4


## 5. Fast exponentiation: FastPower

**Input:** a number $x$ and a natural number $n$. **Output:** $x^n$, by repeated squaring. $S(n) = 4\lfloor\log_2 n\rfloor + 6 = \Theta(\log_2 n)$ steps, $\lfloor\log_2 n\rfloor + 2$ calls, at most $2(\lfloor\log_2 n\rfloor + 1)$ multiplications.

Pseudo-code: section 14.7 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis-recursion/14-recursion/).

In [14]:
def floor_log2(n):                                 # floor(log2 n) with a loop (only for the formula of the steps)
    k = 0
    while n > 1:
        n = n // 2
        k = k + 1
    return k


def fast_power(x, n):
    global mults
    push(n); cnt[2] += 1
    if n == 0:
        cnt[3] += 1
        if trace: say("FastPower(%d, 0)" % x, "base case, returns 1")
        pop()
        return 1
    if trace: say("FastPower(%d, %d)" % (x, n), "%d != 0 => calls FastPower(%d, %d)   (%d/2 = %d)" % (n, x, n // 2, n, n // 2))
    y = fast_power(x, n // 2); cnt[5] += 1         # one recursive call only: the result is in y
    cnt[6] += 1
    if n % 2 == 0:
        cnt[7] += 1
        mults += 1
        r = y * y
        if trace: say("FastPower(%d, %d)" % (x, n), "y = %d, %d is even => returns %d * %d = %d" % (y, n, y, y, r))
    else:
        cnt[9] += 1
        mults += 2
        r = x * y * y
        if trace: say("FastPower(%d, %d)" % (x, n), "y = %d, %d is odd => returns %d * %d * %d = %d" % (y, n, x, y, y, r))
    pop()
    return r

In [15]:
# Example of the lecture notes: FastPower(3, 13)
x, n = 3, 13
reset()
trace = True
print("%-28s| %s" % ("call", "action"))
r = fast_power(x, n)
trace = False
print("%d^%d = %d, calls = %d, multiplications = %d" % (x, n, r, calls, mults))
print_steps(2, 3, 5, 6, 7, 9)
print("S(n) = 4 floor(log2 n) + 6 = %d" % (4 * floor_log2(n) + 6))

# How many multiplications: about 2 log2 n instead of n (x = 1, to avoid overflow)
print()
print("      n |    calls | multiplications | S(n) | 4 floor(log2 n) + 6")
for n in [1, 2, 13, 100, 1000, 1000000]:
    reset()
    fast_power(1, n)
    print("%7d | %8d | %15d | %4d | %d" % (n, calls, mults, sum(cnt), 4 * floor_log2(n) + 6))

call                        | action
FastPower(3, 13)            | 13 != 0 => calls FastPower(3, 6)   (13/2 = 6)
  FastPower(3, 6)           | 6 != 0 => calls FastPower(3, 3)   (6/2 = 3)
    FastPower(3, 3)         | 3 != 0 => calls FastPower(3, 1)   (3/2 = 1)
      FastPower(3, 1)       | 1 != 0 => calls FastPower(3, 0)   (1/2 = 0)
        FastPower(3, 0)     | base case, returns 1
      FastPower(3, 1)       | y = 1, 1 is odd => returns 3 * 1 * 1 = 3
    FastPower(3, 3)         | y = 3, 3 is odd => returns 3 * 3 * 3 = 27
  FastPower(3, 6)           | y = 27, 6 is even => returns 27 * 27 = 729
FastPower(3, 13)            | y = 729, 13 is odd => returns 3 * 729 * 729 = 1594323
3^13 = 1594323, calls = 5, multiplications = 7
Line | Executions
   2 |          5
   3 |          1
   5 |          4
   6 |          4
   7 |          1
   9 |          3
Total steps S = 18
S(n) = 4 floor(log2 n) + 6 = 18

      n |    calls | multiplications | S(n) | 4 floor(log2 n) + 6
      1 |        2 |   

**Try it yourself.** Change $x$ and $n$ and count the multiplications.

In [16]:
x, n = 2, 10        # <-- change x and n (with large x and n C++ and Java overflow)
reset()
trace = True
print("%-28s| %s" % ("call", "action"))
r = fast_power(x, n)
trace = False
print("%d^%d = %d, multiplications = %d" % (x, n, r, mults))

call                        | action
FastPower(2, 10)            | 10 != 0 => calls FastPower(2, 5)   (10/2 = 5)
  FastPower(2, 5)           | 5 != 0 => calls FastPower(2, 2)   (5/2 = 2)
    FastPower(2, 2)         | 2 != 0 => calls FastPower(2, 1)   (2/2 = 1)
      FastPower(2, 1)       | 1 != 0 => calls FastPower(2, 0)   (1/2 = 0)
        FastPower(2, 0)     | base case, returns 1
      FastPower(2, 1)       | y = 1, 1 is odd => returns 2 * 1 * 1 = 2
    FastPower(2, 2)         | y = 2, 2 is even => returns 2 * 2 = 4
  FastPower(2, 5)           | y = 4, 5 is odd => returns 2 * 4 * 4 = 32
FastPower(2, 10)            | y = 32, 10 is even => returns 32 * 32 = 1024
2^10 = 1024, multiplications = 6


## 6. Towers of Hanoi: Hanoi

**Input:** the number of disks $n \ge 1$, the source peg $from$, the destination peg $to$ and the auxiliary peg $aux$. **Output:** the sequence of moves that transfers disks $1, \dots, n$ from $from$ to $to$. $M(n) = 2^n - 1$ moves (optimal), $S(n) = 3 \cdot 2^n - 4 = \Theta(2^n)$ steps, depth $n$.

Pseudo-code: section 14.8 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis-recursion/14-recursion/).

In [17]:
def move(disk, frm, to):                           # "move disk d from from to to": counts the move and, if show_moves, prints it
    global moves
    moves += 1
    if show_moves:
        print("%d. move disk %d from %s to %s" % (moves, disk, frm, to))


show_moves = False


def hanoi(n, frm, to, aux):                        # in Python "from" is a reserved word: frm
    push(n); cnt[2] += 1
    call = "Hanoi(%d,%s,%s,%s)" % (n, frm, to, aux) if trace else ""
    if n == 1:
        cnt[3] += 1
        move(1, frm, to)
        if trace: say(call, "prints: move disk 1 from %s to %s, returns" % (frm, to))
    else:
        if trace: say(call, "calls Hanoi(%d,%s,%s,%s)" % (n - 1, frm, aux, to))
        cnt[5] += 1
        hanoi(n - 1, frm, aux, to)
        cnt[6] += 1
        move(n, frm, to)
        if trace: say(call, "prints: move disk %d from %s to %s, calls Hanoi(%d,%s,%s,%s)" % (n, frm, to, n - 1, aux, to, frm))
        cnt[7] += 1
        hanoi(n - 1, aux, to, frm)
        if trace: say(call, "returns")
    pop()

In [18]:
# Example of the lecture notes: Hanoi(3, A, C, B), the moves
reset()
show_moves = True
hanoi(3, "A", "C", "B")
show_moves = False
print("%d moves, %d calls, depth of the recursion = %d" % (moves, calls, max_depth))

# The same execution, call by call (indentation = depth in the stack)
print()
reset()
trace = True
print("%-28s| %s" % ("call", "action"))
hanoi(3, "A", "C", "B")
trace = False
print_steps(2, 3, 5, 6, 7)
print("S(n) = 3 * 2^n - 4 = %d" % (3 * 2 ** 3 - 4))

# Moves and steps: M(n) = 2^n - 1 and S(n) = 3 * 2^n - 4
print()
print("  n |  moves | 2^n - 1 |    calls |   S(n) | 3 * 2^n - 4 | depth")
for n in range(1, 11):
    reset()
    hanoi(n, "A", "C", "B")
    print("%3d | %6d | %7d | %8d | %6d | %11d | %d" % (n, moves, 2 ** n - 1, calls, sum(cnt), 3 * 2 ** n - 4, max_depth))

# The legend: 64 disks, one move per second
print()
print("n = 64: 2^64 - 1 seconds are more than %.1e years" % ((2.0 ** 64 - 1) / (365.25 * 24 * 3600)))

1. move disk 1 from A to C
2. move disk 2 from A to B
3. move disk 1 from C to B
4. move disk 3 from A to C
5. move disk 1 from B to A
6. move disk 2 from B to C
7. move disk 1 from A to C
7 moves, 7 calls, depth of the recursion = 3

call                        | action
Hanoi(3,A,C,B)              | calls Hanoi(2,A,B,C)
  Hanoi(2,A,B,C)            | calls Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | prints: move disk 1 from A to C, returns
  Hanoi(2,A,B,C)            | prints: move disk 2 from A to B, calls Hanoi(1,C,B,A)
    Hanoi(1,C,B,A)          | prints: move disk 1 from C to B, returns
  Hanoi(2,A,B,C)            | returns
Hanoi(3,A,C,B)              | prints: move disk 3 from A to C, calls Hanoi(2,B,C,A)
  Hanoi(2,B,C,A)            | calls Hanoi(1,B,A,C)
    Hanoi(1,B,A,C)          | prints: move disk 1 from B to A, returns
  Hanoi(2,B,C,A)            | prints: move disk 2 from B to C, calls Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | prints: move disk 1 from A to C, returns
 

**Try it yourself.** Change the number of disks and count the moves.

In [19]:
n = 4        # <-- change the number of disks
reset()
show_moves = True
hanoi(n, "A", "C", "B")
show_moves = False
print("%d moves = 2^%d - 1 = %d" % (moves, n, 2 ** n - 1))

1. move disk 1 from A to B
2. move disk 2 from A to C
3. move disk 1 from B to C
4. move disk 3 from A to B
5. move disk 1 from C to A
6. move disk 2 from C to B
7. move disk 1 from A to B
8. move disk 4 from A to C
9. move disk 1 from B to C
10. move disk 2 from B to A
11. move disk 1 from C to A
12. move disk 3 from B to C
13. move disk 1 from A to B
14. move disk 2 from A to C
15. move disk 1 from B to C
15 moves = 2^4 - 1 = 15
